### Tools 
Models can request to call tools that perform task such as fetching data from a database, searching the web, or running code

Tools are paring of :-

1.A Schema including the name of the tool, a description, and/or arguemnet definition(often a json schema)

2.A function or coroutine to execute.

In [6]:
import langchain
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [8]:
llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

In [9]:
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021AC26E36D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021AC40A4950>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
from langchain.tools import tool

In [24]:
@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"its sunny day in {location}"


model_with_tools = llm.bind_tools([get_weather])

In [25]:
response = model_with_tools.invoke("what is the weather at kanpur")

In [26]:
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'hxq12ystj', 'function': {'arguments': '{"location":"Kanpur"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 219, 'total_tokens': 235, 'completion_time': 0.068884191, 'completion_tokens_details': None, 'prompt_time': 0.025116042, 'prompt_tokens_details': None, 'queue_time': 0.04886715, 'total_time': 0.094000233}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d3e97-0f22-7db2-855d-e48d5e393688-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Kanpur'}, 'id': 'hxq12ystj', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 16, 'total_tokens': 235}


In [46]:
messages = [
    {
        "role": "system",
        "content": "Use tools to fetch data when needed, then summarize the result for the user."
    },
    {
        "role": "user",
        "content": "What is the weather in Kanpur?"
    }
]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)  # ✅ use .content, not .text

The current weather in Kanpur is sunny.


In [49]:
# print(ai_msg)
print(ai_msg.tool_calls)
# print(final_response)

[{'name': 'get_weather', 'args': {'location': 'Kanpur'}, 'id': '4mjnsd2e5', 'type': 'tool_call'}]
